# DERM-Net Multimodal — Image → Diagnosis → Retrieval-Augmented Treatment Report

**One notebook. Press `Run All`.** Needs two inputs attached: the Rare Skin Disease image
dataset and the `Disease, Drug / Active Ingredient` spreadsheet. Both are auto-discovered.

```
  skin photo                                          patient context
      │                                                     │
      ▼                                                     ▼
 ┌──────────────┐   calibrated    ┌───────────────┐   ┌──────────────┐
 │  DERM-Net    │──confidence────▶│  abstention   │──▶│  retrieval   │
 │  (vision)    │   + Grad-CAM    │     gate      │   │ (137 drugs)  │
 └──────────────┘                 └───────────────┘   └──────┬───────┘
                                                             ▼
                                           ┌─────────────────────────────┐
                                           │ deterministic safety engine │
                                           │ pregnancy / CSA / Rx / DDI  │
                                           └──────────────┬──────────────┘
                                                          ▼
                                           ┌─────────────────────────────┐
                                           │  LLM writes the report      │
                                           │  grounded ONLY in retrieved │
                                           └──────────────┬──────────────┘
                                                          ▼
                                           ┌─────────────────────────────┐
                                           │ groundedness verifier       │
                                           │ flags hallucinated drugs    │
                                           └─────────────────────────────┘
```

## Why the drug table is *not* a model input

The obvious move — join the spreadsheet onto the image features and train a "multimodal"
classifier — is **label leakage**, and it is worth being explicit about why.

The table is keyed on `Disease`. Every column (drug, dosage, pregnancy category, …) is a
deterministic function of the diagnosis. Feeding it in as a feature hands the model the answer:
you would see ~100% accuracy and it would mean nothing, because at inference time on a real
patient you do not know which disease's drug row to feed in — that is the thing you are trying
to predict.

The table is only usable **downstream of the prediction**, which is exactly what this notebook
does. Nothing here touches the vision model's training signal.

## What is actually measured

Most "LLM + medical data" demos stop at one nice-looking generated paragraph. This notebook
quantifies the two failure modes that decide whether such a system is safe:

1. **Error propagation.** A vision error becomes a *treatment* error. Every test image is pushed
   through the full chain and scored as correct-treatment / safely-abstained / **wrong-treatment**.
2. **Groundedness.** Every drug name and every dose string in the generated report is checked
   against the retrieved source rows. Anything the LLM invented is counted and flagged.

Plus a **risk–coverage curve**: how much of the caseload you can automate at each confidence
threshold, and what the wrong-treatment rate is at that coverage.

In [ ]:
# =============================================================================
#  CELL 1 - Dependencies
# =============================================================================
import importlib
import subprocess
import sys

_REQUIRED = [
    ("timm", "timm"),
    ("openpyxl", "openpyxl"),
    ("transformers", "transformers"),
    ("pytorch_grad_cam", "grad-cam"),
    ("cv2", "opencv-python-headless"),
    ("tqdm", "tqdm"),
]

_missing = []
for _module, _package in _REQUIRED:
    try:
        importlib.import_module(_module)
    except Exception:
        _missing.append(_package)

if _missing:
    print("Installing:", ", ".join(_missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=False)
    importlib.invalidate_caches()
else:
    print("All dependencies already available.")

# Optional: dense retrieval. Falls back to TF-IDF if unavailable, so this is not fatal.
try:
    importlib.import_module("sentence_transformers")
    print("sentence-transformers available (dense retrieval enabled).")
except Exception:
    print("Installing sentence-transformers (optional, for dense retrieval)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"],
                   check=False)
    importlib.invalidate_caches()

print("Python", sys.version.split()[0])

In [ ]:
# =============================================================================
#  CELL 2 - Imports and configuration
# =============================================================================
import gc
import json
import math
import os
import random
import re
import textwrap
import time
import warnings
from collections import Counter, OrderedDict
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm

import timm

sns.set_style("darkgrid")
plt.rcParams["figure.max_open_warning"] = 0


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class CFG:
    SEED = 42
    IMG_SIZE = 224

    # ---- vision stage -------------------------------------------------------
    # If a dermnet checkpoint is attached as a Kaggle input it is loaded and no
    # training happens. Otherwise DERM-Net is trained here with this schedule.
    TRAIN_EPOCHS = 25
    BATCH_SIZE = 32
    BASE_LR = 1e-4
    WEIGHT_DECAY = 1e-4
    WARMUP_EPOCHS = 2
    LABEL_SMOOTHING = 0.1
    SAMPLER_POWER = 0.5
    EMA_DECAY = 0.999
    USE_AMP = True
    USE_TTA = True

    TEST_SIZE = 0.15
    VAL_SIZE = 0.15
    NUM_WORKERS = 2

    # ---- abstention ---------------------------------------------------------
    # Below this calibrated confidence the system refuses to recommend treatment.
    CONFIDENCE_THRESHOLD = 0.60

    # ---- retrieval ----------------------------------------------------------
    TOP_K = 6
    EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

    # ---- LLM ----------------------------------------------------------------
    # Small instruct models that fit a Kaggle T4/P100 in fp16. First that loads wins.
    LLM_CANDIDATES = [
        "Qwen/Qwen2.5-3B-Instruct",
        "Qwen/Qwen2.5-1.5B-Instruct",
        "microsoft/Phi-3-mini-4k-instruct",
    ]
    LLM_MAX_NEW_TOKENS = 640
    LLM_TEMPERATURE = 0.2

    # ---- evaluation ---------------------------------------------------------
    N_DEMO_CASES = 4        # full reports printed with Grad-CAM figures
    N_EVAL_CASES = 40       # images pushed through the whole chain and scored
    RUN_LLM_EVAL = True     # generate a report for each eval case (slower)


set_seed(CFG.SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = CFG.USE_AMP and DEVICE.type == "cuda"

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
FIG_DIR = OUT_DIR / "figures"
for _d in (OUT_DIR, FIG_DIR):
    _d.mkdir(parents=True, exist_ok=True)


def savefig(fig, name: str) -> None:
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    print(f"   saved -> {path}")


def autocast_ctx(enabled: bool):
    try:
        return torch.amp.autocast("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)


def make_grad_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
print(f"PyTorch {torch.__version__} | timm {timm.__version__}")
print(f"Output : {OUT_DIR}")

## 1. The drug knowledge base

The spreadsheet is found automatically, disease names are reconciled with the image dataset's
class folders (they do not match: `Ichtyosis`/`Ichthyosis`, `Hemangiomas`/`Hemangioma`,
`Port-wine`/`Port-Wine Stains`), and each row becomes one retrievable **drug card**.

In [ ]:
# =============================================================================
#  CELL 3 - Locate and load the drug spreadsheet
# =============================================================================
def find_spreadsheet(search_bases=("/kaggle/input", ".", "./data")):
    """Find an .xlsx/.csv that has a Disease column and a drug column."""
    candidates = []
    for base in search_bases:
        base_path = Path(base)
        if not base_path.is_dir():
            continue
        for pattern in ("**/*.xlsx", "**/*.xls", "**/*.csv"):
            for f in base_path.glob(pattern):
                if f.name.startswith("~$"):
                    continue
                try:
                    head = (pd.read_csv(f, nrows=3) if f.suffix.lower() == ".csv"
                            else pd.read_excel(f, nrows=3))
                except Exception:
                    continue
                cols = {str(c).strip().lower() for c in head.columns}
                if any("disease" in c for c in cols) and any("drug" in c for c in cols):
                    candidates.append((len(head.columns), f))
    if not candidates:
        return None
    candidates.sort(reverse=True)
    return candidates[0][1]


DRUG_FILE = find_spreadsheet()
if DRUG_FILE is None:
    raise FileNotFoundError(
        "Drug spreadsheet not found.\n"
        "On Kaggle: Add Input -> Upload the 'Disease, Drug / Active Ingredient' .xlsx.\n"
        "Locally: put it in the working directory."
    )

drugs_df = (pd.read_csv(DRUG_FILE) if DRUG_FILE.suffix.lower() == ".csv"
            else pd.read_excel(DRUG_FILE))
drugs_df.columns = [str(c).strip() for c in drugs_df.columns]

COL = {
    "disease": "Disease",
    "drug": "Drug / Active Ingredient",
    "brand": "Brand Names / Formulation",
    "cls": "Drug Class / Type",
    "mech": "Mechanism / Use",
    "dose": "Recommended Dosage",
    "side": "Side Effects",
    "ddi": "Key Drug Interactions",
    "preg": "Pregnancy Category",
    "rx": "Rx and OTC status",
    "route": "Route of Administration",
    "csa": "CSA Schedule",
}
# Tolerate small header differences between spreadsheet versions.
for key, expected in list(COL.items()):
    if expected not in drugs_df.columns:
        match = [c for c in drugs_df.columns
                 if expected.lower().split("/")[0].strip() in c.lower()]
        if match:
            COL[key] = match[0]
        else:
            drugs_df[expected] = ""
            COL[key] = expected

for c in COL.values():
    drugs_df[c] = drugs_df[c].fillna("").astype(str).str.strip()

print(f"Spreadsheet : {DRUG_FILE}")
print(f"Rows        : {len(drugs_df)}   Columns: {len(drugs_df.columns)}")
print(f"\nDrugs per disease (as written in the spreadsheet):")
print(drugs_df[COL["disease"]].value_counts().to_string())

In [ ]:
# =============================================================================
#  CELL 4 - Reconcile disease naming between the image classes and the table
# =============================================================================
CANONICAL = {
    "epidermolysisbullosa": "Epidermolysis Bullosa",
    "eb": "Epidermolysis Bullosa",
    "ichtyosis": "Ichthyosis",
    "ichthyosis": "Ichthyosis",
    "hemangioma": "Hemangioma",
    "hemangiomas": "Hemangioma",
    "infantilehemangioma": "Hemangioma",
    "portwine": "Port-Wine Stain",
    "portwinestain": "Port-Wine Stain",
    "portwinestains": "Port-Wine Stain",
    "pws": "Port-Wine Stain",
    "healthyskin": "Healthy Skin",
    "healthy": "Healthy Skin",
    "normal": "Healthy Skin",
    "normalskin": "Healthy Skin",
}

# Diseases with no pharmacological treatment in the table. Predicting one of these
# must never produce a drug recommendation.
NO_TREATMENT = {"Healthy Skin"}


def canon(name: str) -> str:
    key = "".join(ch for ch in str(name).lower() if ch.isalnum())
    return CANONICAL.get(key, str(name).strip())


drugs_df["disease_canon"] = drugs_df[COL["disease"]].map(canon)

print("Disease name reconciliation:\n")
print(f"{'Spreadsheet value':<26} -> {'Canonical':<24} {'Rows':>5}")
print("-" * 60)
for raw, n in drugs_df[COL["disease"]].value_counts().items():
    print(f"{raw:<26} -> {canon(raw):<24} {n:>5}")

KB_DISEASES = sorted(drugs_df["disease_canon"].unique())
print(f"\nDiseases covered by the knowledge base: {KB_DISEASES}")

In [ ]:
# =============================================================================
#  CELL 5 - Build the retrievable drug cards
# =============================================================================
def build_card(row) -> str:
    """One drug row -> one self-contained text document for retrieval and grounding."""
    parts = [
        f"DRUG: {row[COL['drug']]}",
        f"DISEASE: {row['disease_canon']}",
        f"BRAND / FORMULATION: {row[COL['brand']] or 'not specified'}",
        f"CLASS: {row[COL['cls']] or 'not specified'}",
        f"MECHANISM / USE: {row[COL['mech']] or 'not specified'}",
        f"DOSAGE: {row[COL['dose']] or 'not specified'}",
        f"SIDE EFFECTS: {row[COL['side']] or 'not specified'}",
        f"INTERACTIONS: {row[COL['ddi']] or 'none listed'}",
        f"PREGNANCY CATEGORY: {row[COL['preg']] or 'not assigned'}",
        f"STATUS: {row[COL['rx']] or 'not specified'}",
        f"ROUTE: {row[COL['route']] or 'not specified'}",
        f"CONTROLLED SUBSTANCE: {row[COL['csa']] or 'Not Controlled'}",
    ]
    return "\n".join(parts)


drugs_df["card"] = drugs_df.apply(build_card, axis=1)
drugs_df["drug_name"] = drugs_df[COL["drug"]]
drugs_df = drugs_df.reset_index(drop=True)
drugs_df["card_id"] = drugs_df.index

# Every drug/brand string the model is *allowed* to name, for the groundedness check later.
KNOWN_DRUG_NAMES = set()
for _, row in drugs_df.iterrows():
    KNOWN_DRUG_NAMES.add(row["drug_name"].strip())
    for brand in re.split(r"[,/]", row[COL["brand"]]):
        brand = brand.strip()
        if len(brand) > 3:
            KNOWN_DRUG_NAMES.add(brand)
KNOWN_DRUG_NAMES = {n for n in KNOWN_DRUG_NAMES if n}

print(f"Knowledge base : {len(drugs_df)} drug cards")
print(f"Named entities : {len(KNOWN_DRUG_NAMES)} drug + brand names\n")
print("=" * 72)
print("EXAMPLE CARD")
print("=" * 72)
print(drugs_df.iloc[0]["card"])
print("=" * 72)

In [ ]:
# =============================================================================
#  CELL 6 - Deterministic safety engine
# =============================================================================
# These checks are computed in code from the table, never by the LLM. The LLM is
# handed the result and told to reproduce it verbatim. A language model must not
# be the thing that decides whether a drug is safe in pregnancy.

PREGNANCY_UNSAFE = {"X", "D", "AVOID"}
PREGNANCY_CAUTION = {"C"}


def pregnancy_risk(category: str) -> str:
    cat = str(category).strip().upper()
    if not cat or cat.startswith("N/A") or "NOT ASSIGNED" in cat:
        return "unknown"
    head = cat.split("(")[0].strip()
    if head in PREGNANCY_UNSAFE or "AVOID" in cat:
        return "contraindicated"
    if head in PREGNANCY_CAUTION or "D IN 3RD" in cat:
        return "caution"
    if head in {"A", "B"}:
        return "acceptable"
    return "unknown"


def is_controlled(csa: str) -> bool:
    return bool(str(csa).strip()) and "not controlled" not in str(csa).strip().lower()


drugs_df["pregnancy_risk"] = drugs_df[COL["preg"]].map(pregnancy_risk)
drugs_df["controlled"] = drugs_df[COL["csa"]].map(is_controlled)
drugs_df["otc"] = drugs_df[COL["rx"]].str.upper().str.contains("OTC")


def safety_flags(row, context: dict) -> list:
    """Hard safety flags for one drug given a patient context."""
    flags = []
    if context.get("pregnant"):
        risk = row["pregnancy_risk"]
        if risk == "contraindicated":
            flags.append(f"CONTRAINDICATED IN PREGNANCY (category {row[COL['preg']]})")
        elif risk == "caution":
            flags.append(f"pregnancy caution (category {row[COL['preg']]})")
        elif risk == "unknown":
            flags.append("pregnancy safety not established")
    if context.get("infant") and str(row[COL["route"]]).lower().startswith("oral"):
        flags.append("systemic route in an infant - specialist dosing required")
    if row["controlled"]:
        flags.append(f"controlled substance ({row[COL['csa']]}) - misuse and dependence risk")
    interactions = str(row[COL["ddi"]]).strip()
    if interactions and interactions.lower() not in {"none significant", "none known",
                                                     "none listed", "none", "nan"}:
        for med in context.get("current_medications", []):
            if med.strip() and med.strip().lower()[:5] in interactions.lower():
                flags.append(f"INTERACTION with current medication '{med}': {interactions}")
    for allergy in context.get("allergies", []):
        target = f"{row['drug_name']} {row[COL['cls']]}".lower()
        if allergy.strip() and allergy.strip().lower()[:5] in target:
            flags.append(f"ALLERGY match on '{allergy}'")
    return flags


_preg_summary = drugs_df.groupby(["disease_canon", "pregnancy_risk"]).size().unstack(fill_value=0)
print("Pregnancy risk profile of the knowledge base:\n")
print(_preg_summary.to_string())
print(f"\nControlled substances : {int(drugs_df['controlled'].sum())}")
print(f"OTC-available drugs   : {int(drugs_df['otc'].sum())}")
print(f"Rows with a documented interaction: "
      f"{int((~drugs_df[COL['ddi']].str.lower().isin(['', 'none significant', 'none known', 'none listed', 'none'])).sum())}")

In [ ]:
# =============================================================================
#  CELL 7 - Knowledge base overview figure
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Drug knowledge base composition", fontsize=16, fontweight="bold")

counts = drugs_df["disease_canon"].value_counts()
axes[0, 0].bar(range(len(counts)), counts.values,
               color=sns.color_palette("viridis", len(counts)))
axes[0, 0].set_xticks(range(len(counts)))
axes[0, 0].set_xticklabels(counts.index, rotation=20, ha="right", fontsize=9)
axes[0, 0].set_ylabel("Drug entries")
axes[0, 0].set_title("Drugs per disease", fontweight="bold")
for i, v in enumerate(counts.values):
    axes[0, 0].text(i, v + 0.6, str(v), ha="center", fontweight="bold", fontsize=9)

route = drugs_df[COL["route"]].value_counts().head(8)
axes[0, 1].barh(range(len(route)), route.values, color="#42A5F5")
axes[0, 1].set_yticks(range(len(route)))
axes[0, 1].set_yticklabels(route.index, fontsize=9)
axes[0, 1].invert_yaxis()
axes[0, 1].set_title("Route of administration", fontweight="bold")

risk_order = ["acceptable", "caution", "contraindicated", "unknown"]
risk_colors = {"acceptable": "#2E7D32", "caution": "#EF6C00",
               "contraindicated": "#C62828", "unknown": "#9E9E9E"}
bottom = np.zeros(len(counts.index))
for risk in risk_order:
    vals = [int(((drugs_df["disease_canon"] == d) &
                 (drugs_df["pregnancy_risk"] == risk)).sum()) for d in counts.index]
    axes[1, 0].bar(range(len(counts.index)), vals, bottom=bottom,
                   label=risk, color=risk_colors[risk])
    bottom += np.array(vals)
axes[1, 0].set_xticks(range(len(counts.index)))
axes[1, 0].set_xticklabels(counts.index, rotation=20, ha="right", fontsize=9)
axes[1, 0].set_title("Pregnancy risk by disease", fontweight="bold")
axes[1, 0].legend(fontsize=8)

cls = drugs_df[COL["cls"]].value_counts().head(10)
axes[1, 1].barh(range(len(cls)), cls.values, color="#8E24AA")
axes[1, 1].set_yticks(range(len(cls)))
axes[1, 1].set_yticklabels(cls.index, fontsize=8)
axes[1, 1].invert_yaxis()
axes[1, 1].set_title("Most common drug classes", fontweight="bold")

plt.tight_layout()
savefig(fig, "rag_01_knowledge_base")
plt.show()

## 2. Vision stage

Identical architecture and split protocol to the classification notebook. If a DERM-Net
checkpoint is attached as a Kaggle input it is loaded; otherwise it trains here.

In [ ]:
# =============================================================================
#  CELL 8 - Image dataset discovery and the canonical split
# =============================================================================
IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff")


def _count_images(directory: Path) -> int:
    try:
        return sum(1 for f in directory.iterdir()
                   if f.is_file() and f.suffix.lower() in IMG_EXTS)
    except OSError:
        return 0


def find_dataset_root(search_bases=("/kaggle/input", "./data", "."), max_depth: int = 6):
    candidates = []
    for base in search_bases:
        base_path = Path(base)
        if not base_path.is_dir():
            continue
        base_depth = len(base_path.parts)
        for root, dirs, _files in os.walk(base_path):
            root_path = Path(root)
            if len(root_path.parts) - base_depth > max_depth:
                dirs[:] = []
                continue
            dirs[:] = [d for d in dirs if not d.startswith(".")]
            counts = [(root_path / d, _count_images(root_path / d)) for d in dirs]
            counts = [(d, c) for d, c in counts if c >= 5]
            if len(counts) >= 2:
                names = {"".join(ch for ch in d.name.lower() if ch.isalnum())
                         for d, _ in counts}
                if names & {"train", "val", "test", "valid", "training", "testing"}:
                    continue
                candidates.append((len(counts), sum(c for _, c in counts), root_path))
    if not candidates:
        return None
    candidates.sort(key=lambda t: (t[0], t[1]), reverse=True)
    return candidates[0][2]


DATA_ROOT = find_dataset_root()
if DATA_ROOT is None:
    raise FileNotFoundError(
        "Image dataset not found. On Kaggle: Add Input -> 'Rare Skin Disease Dataset'."
    )

CLASS_DIRS = sorted([d for d in DATA_ROOT.iterdir() if d.is_dir() and _count_images(d) >= 5],
                    key=lambda d: d.name.lower())
CLASS_NAMES = [canon(d.name) for d in CLASS_DIRS]
NUM_CLASSES = len(CLASS_NAMES)

FILE_PATHS, FILE_LABELS = [], []
for label, class_dir in enumerate(CLASS_DIRS):
    for f in sorted(class_dir.iterdir()):
        if f.is_file() and f.suffix.lower() in IMG_EXTS:
            FILE_PATHS.append(str(f))
            FILE_LABELS.append(label)
FILE_LABELS = np.asarray(FILE_LABELS)

_indices = np.arange(len(FILE_PATHS))
train_idx, temp_idx = train_test_split(
    _indices, test_size=CFG.TEST_SIZE + CFG.VAL_SIZE,
    stratify=FILE_LABELS, random_state=CFG.SEED)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=CFG.TEST_SIZE / (CFG.TEST_SIZE + CFG.VAL_SIZE),
    stratify=FILE_LABELS[temp_idx], random_state=CFG.SEED)
SPLIT = {"train": np.sort(train_idx), "val": np.sort(val_idx), "test": np.sort(test_idx)}
assert not (set(SPLIT["train"]) & set(SPLIT["test"]))
assert not (set(SPLIT["train"]) & set(SPLIT["val"]))
assert not (set(SPLIT["val"]) & set(SPLIT["test"]))

y_train = FILE_LABELS[SPLIT["train"]]
CLASS_WEIGHTS = torch.tensor(
    compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train),
    dtype=torch.float32)

# Which image classes can actually be treated with this knowledge base?
COVERAGE = {c: (CLASS_NAMES[c] in KB_DISEASES) for c in range(NUM_CLASSES)}

print(f"Image dataset : {DATA_ROOT}")
print(f"Images        : {len(FILE_PATHS)}   Classes: {NUM_CLASSES}")
print(f"Split         : {len(SPLIT['train'])} train / {len(SPLIT['val'])} val / "
      f"{len(SPLIT['test'])} test\n")
print(f"{'Image class':<26} {'Images':>7}   {'Drug cards in KB':>18}")
print("-" * 56)
for c in range(NUM_CLASSES):
    n_cards = int((drugs_df["disease_canon"] == CLASS_NAMES[c]).sum())
    note = "" if COVERAGE[c] else "  <- no drugs, will abstain"
    print(f"{CLASS_NAMES[c]:<26} {int((FILE_LABELS == c).sum()):>7}   {n_cards:>18}{note}")

_uncovered = [CLASS_NAMES[c] for c in range(NUM_CLASSES) if not COVERAGE[c]]
if _uncovered:
    print(f"\nClasses with no pharmacological entry: {_uncovered}")
    print("The pipeline returns an explicit 'no drug treatment indicated' response for these,")
    print("rather than retrieving the nearest-looking drug.")

In [ ]:
# =============================================================================
#  CELL 9 - DERM-Net architecture (same as the classification notebook)
# =============================================================================
class ChannelAttention1D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden, bias=False), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        att = self.sigmoid(self.fc(torch.mean(x, dim=2)) +
                           self.fc(torch.max(x, dim=2).values)).unsqueeze(2)
        return x * att


class MSCABlock1D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        mid = max(1, out_channels // 3)
        self.branch1 = nn.Sequential(nn.Conv1d(in_channels, mid, 1, bias=False),
                                     nn.BatchNorm1d(mid), nn.GELU())
        self.branch3 = nn.Sequential(nn.Conv1d(in_channels, mid, 3, padding=1, bias=False),
                                     nn.BatchNorm1d(mid), nn.GELU())
        self.branch5 = nn.Sequential(nn.Conv1d(in_channels, mid, 5, padding=2, bias=False),
                                     nn.BatchNorm1d(mid), nn.GELU())
        fused_ch = mid * 3
        self.channel_att = ChannelAttention1D(fused_ch)
        self.project = nn.Sequential(nn.Conv1d(fused_ch, out_channels, 1, bias=False),
                                     nn.BatchNorm1d(out_channels))
        self.scale = nn.Parameter(torch.ones(3) / 3)
        self.residual = (nn.Conv1d(in_channels, out_channels, 1, bias=False)
                         if in_channels != out_channels else nn.Identity())
        self.act = nn.GELU()

    def forward(self, x):
        x = x.unsqueeze(2)
        w = F.softmax(self.scale, dim=0)
        fused = torch.cat([self.branch1(x) * w[0], self.branch3(x) * w[1],
                           self.branch5(x) * w[2]], dim=1)
        fused = self.channel_att(fused)
        return self.act(self.project(fused) + self.residual(x)).squeeze(2)


class DERMNet(nn.Module):
    def __init__(self, num_classes, pretrained=True, freeze_vit_blocks=8, dropout=0.4):
        super().__init__()
        self.eff_features = timm.create_model("efficientnet_b4", pretrained=pretrained,
                                              num_classes=0)
        self.vit_features = timm.create_model("vit_base_patch16_224", pretrained=pretrained,
                                              num_classes=0)
        for p in self.vit_features.patch_embed.parameters():
            p.requires_grad = False
        for i in range(min(freeze_vit_blocks, len(self.vit_features.blocks))):
            for p in self.vit_features.blocks[i].parameters():
                p.requires_grad = False
        common_dim = 512
        self.eff_proj = nn.Sequential(
            nn.Linear(self.eff_features.num_features, common_dim),
            nn.LayerNorm(common_dim), nn.GELU())
        self.vit_proj = nn.Sequential(
            nn.Linear(self.vit_features.num_features, common_dim),
            nn.LayerNorm(common_dim), nn.GELU())
        self.fusion_msca = MSCABlock1D(common_dim * 2, common_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(common_dim, 256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes))

    def forward(self, x, return_features=False):
        fused = self.fusion_msca(torch.cat(
            [self.eff_proj(self.eff_features(x)), self.vit_proj(self.vit_features(x))], dim=1))
        fused = self.dropout(fused)
        logits = self.classifier(fused)
        return (logits, fused) if return_features else logits

    def get_eff_target_layer(self):
        return [self.eff_features.conv_head]


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer("weight", weight if weight is not None else None)

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()


class HybridLoss(nn.Module):
    def __init__(self, class_weights, alpha=0.5, beta=0.5):
        super().__init__()
        w = class_weights.to(DEVICE)
        self.focal = FocalLoss(2.0, w).to(DEVICE)
        self.ce = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG.LABEL_SMOOTHING)
        self.alpha, self.beta = alpha, beta

    def forward(self, logits, targets, features=None):
        return self.alpha * self.focal(logits, targets) + self.beta * self.ce(logits, targets)


print("Architecture defined.")

In [ ]:
# =============================================================================
#  CELL 10 - Data loaders
# =============================================================================
class SkinDataset(Dataset):
    def __init__(self, indices, transform=None):
        self.indices = np.asarray(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        gid = int(self.indices[i])
        with Image.open(FILE_PATHS[gid]) as im:
            img = im.convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, int(FILE_LABELS[gid])


TRAIN_TF = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.TrivialAugmentWide(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN.tolist(), IMAGENET_STD.tolist()),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.10), value="random"),
])
EVAL_TF = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN.tolist(), IMAGENET_STD.tolist()),
])


def denormalize(tensor):
    img = tensor.detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(img * IMAGENET_STD + IMAGENET_MEAN, 0, 1).astype(np.float32)


counts = np.bincount(y_train, minlength=NUM_CLASSES).astype(np.float64)
sample_w = ((1.0 / np.maximum(counts, 1)) ** CFG.SAMPLER_POWER)[y_train]
sampler = WeightedRandomSampler(torch.as_tensor(sample_w, dtype=torch.double),
                                len(sample_w), replacement=True)
_common = dict(num_workers=CFG.NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
               persistent_workers=CFG.NUM_WORKERS > 0)
train_loader = DataLoader(SkinDataset(SPLIT["train"], TRAIN_TF),
                          batch_size=CFG.BATCH_SIZE, sampler=sampler, **_common)
val_loader = DataLoader(SkinDataset(SPLIT["val"], EVAL_TF),
                        batch_size=CFG.BATCH_SIZE, shuffle=False, **_common)
test_loader = DataLoader(SkinDataset(SPLIT["test"], EVAL_TF),
                         batch_size=CFG.BATCH_SIZE, shuffle=False, **_common)
print(f"Loaders ready: {len(train_loader)} / {len(val_loader)} / {len(test_loader)} batches")

In [ ]:
# =============================================================================
#  CELL 11 - Load an existing checkpoint, or train the vision stage
# =============================================================================
def find_checkpoint():
    """Look for a DERM-Net .pth in the Kaggle inputs or the working directory."""
    for base in ("/kaggle/input", "/kaggle/working", ".", "./outputs"):
        base_path = Path(base)
        if not base_path.is_dir():
            continue
        for f in base_path.glob("**/*.pth"):
            if "dermnet" in f.name.lower() and "baseline" not in f.name.lower():
                return f
    return None


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float()
                       for k, v in model.state_dict().items() if v.dtype.is_floating_point}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)

    def state_dict(self, model):
        merged = OrderedDict()
        for k, v in model.state_dict().items():
            merged[k] = self.shadow[k].to(v.dtype) if k in self.shadow else v.detach().clone()
        return merged


def _forward(model, x, want_features=False):
    if want_features and isinstance(model, DERMNet):
        return model(x, return_features=True)
    out = model(x)
    if isinstance(out, tuple):
        out = out[0]
    return out, None


def run_epoch(model, loader, criterion, optimizer=None, scheduler=None, scaler=None, ema=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, seen = 0.0, 0, 0
    preds_all, labels_all = [], []
    with (torch.enable_grad() if training else torch.no_grad()):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with autocast_ctx(AMP_ENABLED):
                logits, _ = _forward(model, imgs)
                loss = criterion(logits, labels)
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                if scheduler is not None:
                    scheduler.step()
                if ema is not None:
                    ema.update(model)
            total_loss += loss.item() * imgs.size(0)
            seen += imgs.size(0)
            preds = logits.argmax(1)
            correct += (preds == labels).sum().item()
            preds_all.append(preds.cpu().numpy())
            labels_all.append(labels.cpu().numpy())
    return (total_loss / max(1, seen), correct / max(1, seen),
            f1_score(np.concatenate(labels_all), np.concatenate(preds_all),
                     average="macro", zero_division=0))


CKPT = find_checkpoint()
dermnet = DERMNet(NUM_CLASSES, pretrained=(CKPT is None)).to(DEVICE)

if CKPT is not None:
    try:
        state = torch.load(CKPT, map_location=DEVICE, weights_only=True)
    except TypeError:
        state = torch.load(CKPT, map_location=DEVICE)
    missing, unexpected = dermnet.load_state_dict(state, strict=False)
    print(f"Loaded checkpoint : {CKPT}")
    if missing or unexpected:
        print(f"  (missing {len(missing)}, unexpected {len(unexpected)} keys)")
else:
    print("No checkpoint found - training the vision stage here.\n")
    set_seed(CFG.SEED)
    criterion = HybridLoss(CLASS_WEIGHTS)
    optimizer = torch.optim.AdamW([
        {"params": [p for p in dermnet.vit_features.parameters() if p.requires_grad],
         "lr": CFG.BASE_LR * 0.1},
        {"params": dermnet.eff_features.parameters(), "lr": CFG.BASE_LR},
        {"params": dermnet.eff_proj.parameters(), "lr": CFG.BASE_LR * 5},
        {"params": dermnet.vit_proj.parameters(), "lr": CFG.BASE_LR * 5},
        {"params": dermnet.fusion_msca.parameters(), "lr": CFG.BASE_LR * 5},
        {"params": dermnet.classifier.parameters(), "lr": CFG.BASE_LR * 5},
    ], weight_decay=CFG.WEIGHT_DECAY)

    steps = max(1, len(train_loader))
    warmup = max(1, CFG.WARMUP_EPOCHS * steps)
    total = max(warmup + 1, CFG.TRAIN_EPOCHS * steps)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda s: (s + 1) / warmup if s < warmup
        else 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warmup) / max(1, total - warmup)))))
    scaler = make_grad_scaler(AMP_ENABLED)
    ema = ModelEMA(dermnet, CFG.EMA_DECAY)

    best_f1, best_state = -1.0, None
    print(f"{'Ep':>3} | {'TrLoss':>8} {'TrF1':>6} | {'VaLoss':>8} {'VaAcc':>7} {'VaF1':>6} | {'src':>4}")
    print("-" * 66)
    for epoch in range(1, CFG.TRAIN_EPOCHS + 1):
        tr_loss, _, tr_f1 = run_epoch(dermnet, train_loader, criterion,
                                      optimizer, scheduler, scaler, ema)
        va_loss, va_acc, va_f1 = run_epoch(dermnet, val_loader, criterion)
        tag, cand = "raw", {k: v.detach().clone() for k, v in dermnet.state_dict().items()}
        live = {k: v.detach().clone() for k, v in dermnet.state_dict().items()}
        dermnet.load_state_dict(ema.state_dict(dermnet))
        e_loss, e_acc, e_f1 = run_epoch(dermnet, val_loader, criterion)
        if e_f1 > va_f1:
            va_loss, va_acc, va_f1, tag = e_loss, e_acc, e_f1, "ema"
            cand = {k: v.detach().clone() for k, v in dermnet.state_dict().items()}
        dermnet.load_state_dict(live)
        mark = ""
        if va_f1 > best_f1:
            best_f1, best_state, mark = va_f1, cand, "  <- best"
        print(f"{epoch:>3} | {tr_loss:>8.4f} {tr_f1:>6.3f} | {va_loss:>8.4f} "
              f"{va_acc * 100:>6.2f}% {va_f1:>6.3f} | {tag:>4}{mark}")
    if best_state is not None:
        dermnet.load_state_dict(best_state)
    torch.save(dermnet.state_dict(), OUT_DIR / "dermnet_best.pth")
    print(f"\nBest val macro-F1 {best_f1:.4f} | saved -> {OUT_DIR / 'dermnet_best.pth'}")

dermnet.eval()
print("Vision stage ready.")

In [ ]:
# =============================================================================
#  CELL 12 - Calibration and test-set predictions
# =============================================================================
@torch.no_grad()
def collect_logits(model, loader):
    model.eval()
    logits_all, labels_all = [], []
    for imgs, labels in loader:
        with autocast_ctx(AMP_ENABLED):
            logits, _ = _forward(model, imgs.to(DEVICE))
        logits_all.append(logits.float().cpu())
        labels_all.append(labels)
    return torch.cat(logits_all), torch.cat(labels_all)


def fit_temperature(model, loader) -> float:
    logits, labels = collect_logits(model, loader)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / torch.exp(log_t), labels)
        loss.backward()
        return loss

    try:
        opt.step(closure)
        t = float(torch.exp(log_t).item())
    except Exception:
        return 1.0
    return float(np.clip(t, 0.05, 10.0)) if np.isfinite(t) else 1.0


@torch.no_grad()
def predict_image_tensor(img_tensor: torch.Tensor, temperature: float = 1.0,
                         tta: bool = True) -> np.ndarray:
    """Single normalized image tensor (C,H,W) -> calibrated probability vector."""
    dermnet.eval()
    batch = img_tensor.unsqueeze(0).to(DEVICE)
    views = [batch]
    if tta:
        views += [torch.flip(batch, dims=[3]), torch.flip(batch, dims=[2]),
                  torch.flip(batch, dims=[2, 3])]
    probs = None
    for view in views:
        with autocast_ctx(AMP_ENABLED):
            logits, _ = _forward(dermnet, view)
        p = F.softmax(logits.float() / temperature, dim=1)
        probs = p if probs is None else probs + p
    return (probs / len(views)).cpu().numpy()[0]


TEMPERATURE = fit_temperature(dermnet, val_loader)
print(f"Calibration temperature T = {TEMPERATURE:.3f}")

test_ds = SkinDataset(SPLIT["test"], EVAL_TF)
test_probs, test_labels = [], []
for i in tqdm(range(len(test_ds)), desc="Test predictions"):
    img_t, lbl = test_ds[i]
    test_probs.append(predict_image_tensor(img_t, TEMPERATURE, CFG.USE_TTA))
    test_labels.append(lbl)
test_probs = np.stack(test_probs)
test_labels = np.asarray(test_labels)
test_preds = test_probs.argmax(axis=1)

vision_acc = float((test_preds == test_labels).mean())
vision_f1 = f1_score(test_labels, test_preds, average="macro", zero_division=0)
print(f"\nVision stage on the test split: accuracy {vision_acc * 100:.2f}%  "
      f"macro-F1 {vision_f1:.4f}")
print(f"Mean confidence: {test_probs.max(axis=1).mean():.3f}   "
      f"above threshold ({CFG.CONFIDENCE_THRESHOLD}): "
      f"{(test_probs.max(axis=1) >= CFG.CONFIDENCE_THRESHOLD).mean() * 100:.1f}% of cases")

## 3. Retrieval

Two stages, in this order:

1. **Structured filter** on the predicted disease. This is not optional — recommending an
   ichthyosis keratolytic for a haemangioma because the free text looked similar would be a
   serious error, and a pure embedding search will happily do that.
2. **Ranking within the disease** by relevance to the patient context (pregnancy, infant,
   infection, pain, …), using dense embeddings when `sentence-transformers` is available and
   TF-IDF otherwise. Both paths are exercised, so the notebook never hard-fails on this.

In [ ]:
# =============================================================================
#  CELL 13 - Retrieval engine
# =============================================================================
class DrugRetriever:
    def __init__(self, dataframe: pd.DataFrame):
        self.df = dataframe
        self.cards = dataframe["card"].tolist()
        self.backend = "tfidf"
        self.encoder = None
        try:
            from sentence_transformers import SentenceTransformer
            self.encoder = SentenceTransformer(CFG.EMBED_MODEL, device=str(DEVICE))
            self.embeddings = self.encoder.encode(
                self.cards, convert_to_numpy=True, normalize_embeddings=True,
                show_progress_bar=False, batch_size=32)
            self.backend = "dense"
        except Exception as exc:
            print(f"Dense retrieval unavailable ({type(exc).__name__}); using TF-IDF.")
            self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2),
                                              sublinear_tf=True)
            self.matrix = self.vectorizer.fit_transform(self.cards)
        print(f"Retriever ready over {len(self.cards)} cards (backend: {self.backend}).")

    def _scores(self, query: str) -> np.ndarray:
        if self.backend == "dense":
            q = self.encoder.encode([query], convert_to_numpy=True,
                                    normalize_embeddings=True, show_progress_bar=False)
            return (self.embeddings @ q[0])
        q = self.vectorizer.transform([query])
        return np.asarray((self.matrix @ q.T).todense()).ravel()

    def retrieve(self, disease: str, query: str, k: int = 6) -> pd.DataFrame:
        """Structured filter on disease, then rank the survivors by the query."""
        mask = (self.df["disease_canon"] == disease).values
        if mask.sum() == 0:
            return self.df.iloc[[]].copy()
        scores = self._scores(query)
        scores = np.where(mask, scores, -np.inf)
        top = np.argsort(scores)[::-1][:min(k, int(mask.sum()))]
        out = self.df.iloc[top].copy()
        out["retrieval_score"] = scores[top]
        return out


retriever = DrugRetriever(drugs_df)


def build_query(disease: str, context: dict) -> str:
    """Turn the structured patient context into a retrieval query."""
    bits = [f"treatment for {disease}"]
    if context.get("pregnant"):
        bits.append("safe in pregnancy, avoid teratogenic retinoid")
    if context.get("infant"):
        bits.append("infant, paediatric dosing, topical preferred")
    if context.get("infected"):
        bits.append("wound infection, antibiotic, antimicrobial dressing")
    if context.get("pain"):
        bits.append("pain relief, analgesia, anaesthetic")
    if context.get("severe"):
        bits.append("severe extensive disease, systemic therapy, biologic")
    if context.get("prefer_otc"):
        bits.append("over the counter, emollient, no prescription")
    for note in context.get("notes", []):
        bits.append(str(note))
    return "; ".join(bits)


# Sanity check: does the structured filter actually hold?
_demo_ctx = {"pregnant": True}
_demo = retriever.retrieve("Ichthyosis", build_query("Ichthyosis", _demo_ctx), k=5)
print(f"\nExample retrieval - Ichthyosis, pregnant patient (backend {retriever.backend}):")
for _, r in _demo.iterrows():
    print(f"  {r['drug_name']:<34} {r[COL['cls']]:<28} "
          f"pregnancy={r[COL['preg']] or 'n/a':<6} risk={r['pregnancy_risk']}")
assert (_demo["disease_canon"] == "Ichthyosis").all(), "structured disease filter leaked"
print("\nStructured filter verified: every retrieved card belongs to the queried disease.")

## 4. The language model

A small instruct model runs locally on the Kaggle GPU — no API key, no cost, no data leaving
the notebook (which matters for clinical text).

If no model can be loaded, the pipeline falls back to a **deterministic template generator**
rather than crashing. That fallback is not a toy: it is grounded in the retrieved cards by
construction, so the whole evaluation harness downstream still produces meaningful numbers, and
it gives the LLM a floor to be compared against.

In [ ]:
# =============================================================================
#  CELL 14 - LLM backend with a deterministic fallback
# =============================================================================
class TemplateLLM:
    """Deterministic, grounded-by-construction report writer. The fallback and the baseline."""

    name = "template-baseline"
    is_llm = False

    def generate(self, system_prompt: str, user_prompt: str, cards: pd.DataFrame,
                 header: dict) -> str:
        lines = [
            f"ASSESSMENT",
            f"Image-based impression: {header['disease']} "
            f"(model confidence {header['confidence'] * 100:.1f}%).",
            "",
            "TREATMENT OPTIONS (from the supplied formulary only)",
        ]
        for i, (_, r) in enumerate(cards.iterrows(), 1):
            lines.append(
                f"{i}. {r['drug_name']} ({r[COL['brand']] or 'no brand listed'}) - "
                f"{r[COL['cls']]}, {r[COL['route']].lower()} route.")
            lines.append(f"   Use: {r[COL['mech']]}")
            lines.append(f"   Dosage: {r[COL['dose']]}")
            lines.append(f"   Side effects: {r[COL['side']]}")
            lines.append(f"   Status: {r[COL['rx']]}; pregnancy category "
                         f"{r[COL['preg']] or 'not assigned'}.")
        lines.append("")
        lines.append("SAFETY")
        if header["flags"]:
            for flag in header["flags"]:
                lines.append(f"- {flag}")
        else:
            lines.append("- No hard contraindication identified for the supplied context.")
        lines.append("")
        lines.append("This is decision support based on an image model and a fixed formulary. "
                     "It is not a diagnosis. A clinician must confirm before any treatment.")
        return "\n".join(lines)


class LocalLLM:
    """A small instruct model from transformers, run on the notebook's GPU."""

    is_llm = True

    def __init__(self, model_id: str):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.name = model_id
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
            device_map="auto" if DEVICE.type == "cuda" else None,
            trust_remote_code=True,
        )
        self.model.eval()
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    @torch.no_grad()
    def generate(self, system_prompt: str, user_prompt: str, cards=None, header=None) -> str:
        messages = [{"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}]
        try:
            text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            text = f"{system_prompt}\n\n{user_prompt}\n\nReport:\n"
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        output = self.model.generate(
            **inputs,
            max_new_tokens=CFG.LLM_MAX_NEW_TOKENS,
            do_sample=CFG.LLM_TEMPERATURE > 0,
            temperature=max(CFG.LLM_TEMPERATURE, 1e-5),
            top_p=0.9,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        generated = output[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(generated, skip_special_tokens=True).strip()


def load_llm():
    for model_id in CFG.LLM_CANDIDATES:
        try:
            print(f"Loading {model_id} ...")
            llm = LocalLLM(model_id)
            print(f"Loaded: {model_id}")
            return llm
        except Exception as exc:
            print(f"  could not load ({type(exc).__name__}: {str(exc)[:140]})")
    print("\nNo LLM could be loaded - using the deterministic template generator.")
    print("On Kaggle this usually means Internet is off (Notebook options -> Internet ON)")
    print("or the GPU accelerator is not enabled.")
    return TemplateLLM()


LLM = load_llm()
TEMPLATE_LLM = TemplateLLM()
print(f"\nGenerator in use: {LLM.name}")

In [ ]:
# =============================================================================
#  CELL 15 - Prompt construction
# =============================================================================
SYSTEM_PROMPT = textwrap.dedent("""
    You are a clinical decision-support assistant for a dermatology triage service.

    Absolute rules:
    1. Use ONLY the drug information provided in the CONTEXT block. It is the complete
       formulary available to you.
    2. Never name a drug that does not appear in the CONTEXT.
    3. Never state a dose, frequency or duration that does not appear in the CONTEXT.
       Copy dosing text exactly as written.
    4. Reproduce every line of the SAFETY block verbatim in your Safety section. Do not
       soften, reword or omit a contraindication.
    5. The image model gives an impression, not a diagnosis. Say so.
    6. If the CONTEXT is empty, state that no pharmacological treatment is available and
       recommend specialist referral. Do not suggest anything from your own knowledge.

    Write in this structure, plain text, no markdown:

    ASSESSMENT
    <two sentences: the impression and the model confidence>

    TREATMENT OPTIONS
    <numbered list; for each: drug name, class, route, what it is for, the exact dosage
     text from the context, main side effects, prescription status>

    SAFETY
    <the verbatim safety lines, then any pregnancy/controlled-substance notes from the context>

    NEXT STEPS
    <what the clinician should confirm; when to refer urgently>
""").strip()


def build_user_prompt(disease, confidence, top3, cards, context, flags) -> str:
    if len(cards) == 0:
        context_block = "(no drug entries available for this condition)"
    else:
        context_block = "\n\n".join(f"--- CARD {i + 1} ---\n{c}"
                                    for i, c in enumerate(cards["card"].tolist()))

    ctx_lines = []
    if context.get("pregnant"):
        ctx_lines.append("Patient is pregnant.")
    if context.get("infant"):
        ctx_lines.append("Patient is an infant.")
    if context.get("infected"):
        ctx_lines.append("Wound shows signs of infection.")
    if context.get("pain"):
        ctx_lines.append("Patient reports significant pain.")
    if context.get("severe"):
        ctx_lines.append("Disease is extensive or severe.")
    if context.get("prefer_otc"):
        ctx_lines.append("Patient has no prescription access; prefers OTC options.")
    for med in context.get("current_medications", []):
        ctx_lines.append(f"Current medication: {med}.")
    for allergy in context.get("allergies", []):
        ctx_lines.append(f"Known allergy: {allergy}.")
    if not ctx_lines:
        ctx_lines.append("No additional clinical context supplied.")

    differential = ", ".join(f"{CLASS_NAMES[c]} {p * 100:.1f}%" for c, p in top3)
    safety_block = "\n".join(f"- {f}" for f in flags) if flags else \
        "- No hard contraindication identified for the supplied context."

    return textwrap.dedent(f"""
        IMAGE MODEL OUTPUT
        Impression: {disease}
        Calibrated confidence: {confidence * 100:.1f}%
        Differential: {differential}

        PATIENT CONTEXT
        {chr(10).join('- ' + line for line in ctx_lines)}

        SAFETY (computed deterministically - reproduce verbatim)
        {safety_block}

        CONTEXT (the only drugs you may name)
        {context_block}

        Write the report now.
    """).strip()


print("Prompt template ready.")
print(f"System prompt: {len(SYSTEM_PROMPT)} chars, {len(SYSTEM_PROMPT.split())} words")

## 5. The pipeline, and the verifier that checks it

`run_pipeline` is the whole system in one function. `verify_report` is what makes it
trustworthy: it re-reads the generated text and checks every drug name and every dose string
against the cards that were actually retrieved.

In [ ]:
# =============================================================================
#  CELL 16 - Groundedness verifier
# =============================================================================
DOSE_PATTERN = re.compile(
    r"\d+(?:\.\d+)?\s*(?:mg|mcg|µg|g|ml|mL|%|IU|units?)(?:\s*/\s*kg)?(?:\s*/\s*(?:day|dose))?",
    re.IGNORECASE)

# A percentage is only a *dose* if it is a concentration. Reports also state model
# confidence ("87.3% confidence"), which must not be scored as an ungrounded dose -
# it is a pipeline output, not a claim about the formulary.
NON_DOSE_CONTEXT = re.compile(
    r"confidence|probability|certainty|threshold|accuracy|likelihood|differential|impression",
    re.IGNORECASE)


def extract_doses(report: str) -> list:
    """Dose-like strings, excluding percentages that are really confidence figures."""
    doses = []
    for match in DOSE_PATTERN.finditer(report):
        token = match.group().strip()
        window = report[max(0, match.start() - 45):match.end() + 30]
        if token.endswith("%") and NON_DOSE_CONTEXT.search(window):
            continue
        doses.append(token)
    return doses


def _mentions(name: str, text: str) -> bool:
    """Whole-word-ish containment that tolerates punctuation inside drug names."""
    name = name.strip()
    if len(name) < 4:
        return False
    pattern = r"(?<![A-Za-z])" + re.escape(name) + r"(?![A-Za-z])"
    return re.search(pattern, text, re.IGNORECASE) is not None


def allowed_names(cards: pd.DataFrame) -> set:
    names = set()
    for _, r in cards.iterrows():
        names.add(str(r["drug_name"]).strip())
        for brand in re.split(r"[,/]", str(r[COL["brand"]])):
            if len(brand.strip()) > 3:
                names.add(brand.strip())
    return {n for n in names if n}


def _norm_space(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip().lower()


def verify_report(report: str, cards: pd.DataFrame, flags: list) -> dict:
    """Check the generated text against the retrieved source cards."""
    permitted = allowed_names(cards)
    source_raw = " ".join(cards["card"].tolist()) if len(cards) else ""
    source_text = _norm_space(source_raw)

    cited = {n for n in permitted if _mentions(n, report)}
    # A hallucination is a drug named in the report that appears NOWHERE in the supplied
    # context. Drug cards quote other drugs inside their INTERACTIONS field ("Chloramphenicol
    # (potential antagonism)"), so a faithful report that copies that field names a drug it
    # did not retrieve - that is grounded, not invented, and must not be flagged.
    out_of_context = {
        n for n in KNOWN_DRUG_NAMES - permitted
        if _mentions(n, report) and not _mentions(n, source_raw)
    }

    doses = extract_doses(report)
    grounded_doses = [d for d in doses if _norm_space(d) in source_text]
    # No dose claimed = nothing to be wrong about. An abstention report is vacuously grounded.
    dose_groundedness = (len(grounded_doses) / len(doses)) if doses else 1.0

    reproduced = 0
    for flag in flags:
        key = _norm_space(flag)[:40]
        if key and key in _norm_space(report):
            reproduced += 1
    safety_reproduction = (reproduced / len(flags)) if flags else 1.0

    return {
        "drugs_cited": len(cited),
        "drugs_available": len(permitted),
        "hallucinated_drugs": sorted(out_of_context),
        "n_hallucinated": len(out_of_context),
        "doses_stated": len(doses),
        "doses_grounded": len(grounded_doses),
        "dose_groundedness": dose_groundedness,
        "safety_reproduction": safety_reproduction,
        "is_grounded": len(out_of_context) == 0 and dose_groundedness == 1.0,
    }


# Self-test with a deliberately hallucinated report.
_test_cards = drugs_df[drugs_df["disease_canon"] == "Ichthyosis"].head(3)
_good = " ".join(_test_cards["card"].tolist())
_bad = _good + "\nAlso start Propranolol 2 mg/kg/day and Isotretinoin 999 mg daily."
_v_good = verify_report(_good, _test_cards, [])
_v_bad = verify_report(_bad, _test_cards, [])
print("Verifier self-test")
print(f"  faithful text  -> hallucinated={_v_bad and _v_good['n_hallucinated']}, "
      f"dose groundedness={_v_good['dose_groundedness']:.2f}, "
      f"grounded={_v_good['is_grounded']}")
print(f"  tampered text  -> hallucinated={_v_bad['n_hallucinated']} "
      f"{_v_bad['hallucinated_drugs']}, "
      f"dose groundedness={_v_bad['dose_groundedness']:.2f}, "
      f"grounded={_v_bad['is_grounded']}")
assert _v_good["is_grounded"], "verifier rejected a faithful report"
assert not _v_bad["is_grounded"], "verifier failed to catch an injected hallucination"
assert _v_bad["n_hallucinated"] >= 1, "injected out-of-formulary drug was not detected"

# A confidence percentage is not a dose. Without this, every abstention report would
# be scored as ungrounded because "42.9% confidence" looks like a concentration.
_conf_text = ("ASSESSMENT\nImpression: Ichthyosis at 87.3% confidence, below the 60% "
              "threshold. No treatment recommended; refer for assessment.")
_v_conf = verify_report(_conf_text, drugs_df.iloc[[]], [])
assert _v_conf["doses_stated"] == 0, "confidence percentage misread as a dose"
assert _v_conf["is_grounded"], "abstention report should be vacuously grounded"
print(f"  abstention text -> doses detected={_v_conf['doses_stated']}, "
      f"grounded={_v_conf['is_grounded']} (confidence % correctly ignored)")
print("  verifier behaves correctly on all three cases.")

In [ ]:
# =============================================================================
#  CELL 17 - The end-to-end pipeline
# =============================================================================
def run_pipeline(img_tensor, context=None, generator=None, true_label=None,
                 k=None, verbose=False) -> dict:
    """Image + patient context -> retrieved drugs, safety flags, report, verification."""
    context = context or {}
    generator = generator or LLM
    k = k or CFG.TOP_K

    probs = predict_image_tensor(img_tensor, TEMPERATURE, CFG.USE_TTA)
    pred = int(probs.argmax())
    confidence = float(probs[pred])
    disease = CLASS_NAMES[pred]
    order = np.argsort(probs)[::-1][:3]
    top3 = [(int(c), float(probs[c])) for c in order]

    # ---- decision gate ------------------------------------------------------
    if disease in NO_TREATMENT:
        status = "no_treatment_indicated"
    elif confidence < CFG.CONFIDENCE_THRESHOLD:
        status = "abstained_low_confidence"
    elif disease not in KB_DISEASES:
        status = "no_kb_coverage"
    else:
        status = "recommended"

    if status == "recommended":
        query = build_query(disease, context)
        cards = retriever.retrieve(disease, query, k=k)
        flags = []
        for _, row in cards.iterrows():
            for flag in safety_flags(row, context):
                entry = f"{row['drug_name']}: {flag}"
                if entry not in flags:
                    flags.append(entry)
    else:
        cards = drugs_df.iloc[[]].copy()
        flags = []
        if status == "abstained_low_confidence":
            flags.append(
                f"Model confidence {confidence * 100:.1f}% is below the "
                f"{CFG.CONFIDENCE_THRESHOLD * 100:.0f}% threshold - no treatment recommended, "
                f"refer for clinical assessment.")
        elif status == "no_treatment_indicated":
            flags.append("Image impression is healthy skin - no pharmacological treatment "
                         "indicated.")
        else:
            flags.append(f"No formulary entry exists for {disease} - specialist referral.")

    user_prompt = build_user_prompt(disease, confidence, top3, cards, context, flags)
    t0 = time.time()
    report = generator.generate(SYSTEM_PROMPT, user_prompt, cards, {
        "disease": disease, "confidence": confidence, "flags": flags})
    gen_seconds = time.time() - t0

    verification = verify_report(report, cards, flags)

    result = {
        "pred": pred, "disease": disease, "confidence": confidence, "top3": top3,
        "status": status, "cards": cards, "flags": flags, "report": report,
        "verification": verification, "gen_seconds": gen_seconds,
        "generator": generator.name, "probs": probs,
    }
    if true_label is not None:
        result["true_label"] = int(true_label)
        result["true_disease"] = CLASS_NAMES[int(true_label)]
        result["vision_correct"] = int(true_label) == pred
        if status == "recommended":
            result["outcome"] = ("correct_treatment" if result["vision_correct"]
                                 else "wrong_treatment")
        elif CLASS_NAMES[int(true_label)] in NO_TREATMENT:
            result["outcome"] = ("correct_no_treatment"
                                 if status == "no_treatment_indicated" else "safe_abstention")
        else:
            result["outcome"] = "missed_treatment"

    if verbose:
        print_case(result)
    return result


def print_case(result: dict) -> None:
    width = 84
    print("=" * width)
    header = f"  IMPRESSION: {result['disease']}   ({result['confidence'] * 100:.1f}% confidence)"
    if "true_disease" in result:
        mark = "correct" if result["vision_correct"] else "INCORRECT"
        header += f"   [ground truth: {result['true_disease']} - {mark}]"
    print(header)
    print(f"  Status: {result['status']}   |   generator: {result['generator']}   "
          f"|   {result['gen_seconds']:.1f}s")
    print("=" * width)
    print("  Differential: " + ", ".join(
        f"{CLASS_NAMES[c]} {p * 100:.1f}%" for c, p in result["top3"]))
    if len(result["cards"]):
        print(f"  Retrieved {len(result['cards'])} drug cards: " +
              ", ".join(result["cards"]["drug_name"].tolist()))
    print("-" * width)
    print(result["report"])
    print("-" * width)
    v = result["verification"]
    verdict = "GROUNDED" if v["is_grounded"] else "UNGROUNDED"
    print(f"  Verification: {verdict}  |  drugs cited {v['drugs_cited']}/{v['drugs_available']}"
          f"  |  doses grounded {v['doses_grounded']}/{v['doses_stated']}"
          f"  |  safety reproduced {v['safety_reproduction'] * 100:.0f}%")
    if v["hallucinated_drugs"]:
        print(f"  HALLUCINATED DRUGS: {', '.join(v['hallucinated_drugs'])}")
    print("=" * width)


print("Pipeline ready.")

In [ ]:
# =============================================================================
#  CELL 18 - Worked demonstration cases
# =============================================================================
try:
    from pytorch_grad_cam import GradCAMPlusPlus
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    CAM = GradCAMPlusPlus(model=dermnet, target_layers=dermnet.get_eff_target_layer())
    CAM_OK = True
except Exception as exc:
    print(f"Grad-CAM unavailable ({exc}); demo will show images without heatmaps.")
    CAM_OK = False

# One demo per class, plus a deliberately varied patient context for each so the
# safety engine is visibly exercised.
DEMO_CONTEXTS = [
    {"pregnant": True, "notes": ["second trimester"]},
    {"infant": True, "notes": ["4 months old"]},
    {"infected": True, "pain": True, "notes": ["painful blistering, purulent discharge"]},
    {"prefer_otc": True, "notes": ["rural clinic, no prescriber on site"]},
    {"severe": True, "current_medications": ["Methotrexate"], "allergies": ["Sulfa"]},
]

demo_indices = []
for c in range(NUM_CLASSES):
    pool = np.flatnonzero(test_labels == c)
    if len(pool):
        demo_indices.append(int(pool[np.argmax(test_probs[pool].max(axis=1))]))
demo_indices = demo_indices[:max(1, CFG.N_DEMO_CASES)]

demo_results = []
for n, idx in enumerate(demo_indices):
    img_t, lbl = test_ds[idx]
    ctx = DEMO_CONTEXTS[n % len(DEMO_CONTEXTS)]
    print(f"\n\n{'#' * 84}")
    print(f"#  DEMONSTRATION CASE {n + 1}   patient context: "
          f"{ {k: v for k, v in ctx.items() if v} }")
    print(f"{'#' * 84}")
    res = run_pipeline(img_t, context=ctx, true_label=lbl, verbose=True)
    res["_idx"] = idx
    res["_context"] = ctx
    demo_results.append(res)

# ---- figure: image + Grad-CAM + retrieved drugs -----------------------------
n_rows = len(demo_results)
fig, axes = plt.subplots(n_rows, 3, figsize=(17, 4.6 * n_rows),
                         gridspec_kw={"width_ratios": [1, 1, 2.1]})
axes = np.atleast_2d(axes)
fig.suptitle("Image → diagnosis → retrieved formulary, per demonstration case",
             fontsize=16, fontweight="bold", y=1.002)

for row, res in enumerate(demo_results):
    img_t, _ = test_ds[res["_idx"]]
    img_np = denormalize(img_t)

    axes[row, 0].imshow(img_np)
    axes[row, 0].axis("off")
    axes[row, 0].set_title(f"true: {res['true_disease']}", fontsize=11)

    if CAM_OK:
        try:
            cam_map = CAM(input_tensor=img_t.unsqueeze(0).to(DEVICE),
                          targets=[ClassifierOutputTarget(res["pred"])],
                          eigen_smooth=True)[0]
            lo, hi = cam_map.min(), cam_map.max()
            cam_map = (cam_map - lo) / (hi - lo + 1e-8)
            axes[row, 1].imshow(show_cam_on_image(img_np, cam_map, use_rgb=True))
        except Exception:
            axes[row, 1].imshow(img_np)
    else:
        axes[row, 1].imshow(img_np)
    axes[row, 1].axis("off")
    colour = "darkgreen" if res.get("vision_correct") else "darkred"
    axes[row, 1].set_title(f"{res['disease']} ({res['confidence'] * 100:.1f}%)",
                           fontsize=11, color=colour)

    ax = axes[row, 2]
    ax.axis("off")
    if len(res["cards"]):
        text_lines = [f"Retrieved formulary ({res['status']}):", ""]
        for i, (_, r) in enumerate(res["cards"].head(5).iterrows(), 1):
            text_lines.append(f"{i}. {r['drug_name']}  [{r[COL['route']]}]")
            text_lines.append(f"    {r[COL['cls']]} | pregnancy {r[COL['preg']] or 'n/a'} "
                              f"| {r[COL['rx']]}")
    else:
        text_lines = [f"No drugs retrieved", "", f"status: {res['status']}"]
    if res["flags"]:
        text_lines += ["", "Safety flags:"]
        for f in res["flags"][:4]:
            text_lines.append("  - " + textwrap.shorten(f, 74))
    ax.text(0, 1, "\n".join(text_lines), va="top", ha="left", fontsize=8.5,
            family="monospace", transform=ax.transAxes)

plt.tight_layout()
savefig(fig, "rag_02_demo_cases")
plt.show()

## 6. End-to-end evaluation

The number that matters is not vision accuracy and not report fluency. It is: **how often does
the complete system hand a clinician the wrong treatment?**

In [ ]:
# =============================================================================
#  CELL 19 - Push the test split through the whole chain
# =============================================================================
rng = np.random.RandomState(CFG.SEED)
n_eval = min(CFG.N_EVAL_CASES, len(test_ds))
eval_indices = rng.choice(len(test_ds), size=n_eval, replace=False)

EVAL_CONTEXTS = DEMO_CONTEXTS + [{}, {}, {"pain": True}, {"infected": True}]

records = []
for n, idx in enumerate(tqdm(eval_indices, desc="End-to-end evaluation")):
    img_t, lbl = test_ds[int(idx)]
    ctx = EVAL_CONTEXTS[n % len(EVAL_CONTEXTS)]
    generator = LLM if CFG.RUN_LLM_EVAL else TEMPLATE_LLM
    res = run_pipeline(img_t, context=ctx, generator=generator, true_label=lbl)
    v = res["verification"]
    retrieved_correct = (
        float((res["cards"]["disease_canon"] == res["true_disease"]).mean())
        if len(res["cards"]) else np.nan)
    records.append({
        "idx": int(idx),
        "true_disease": res["true_disease"],
        "pred_disease": res["disease"],
        "confidence": res["confidence"],
        "vision_correct": res["vision_correct"],
        "status": res["status"],
        "outcome": res["outcome"],
        "n_retrieved": len(res["cards"]),
        "retrieval_precision": retrieved_correct,
        "n_flags": len(res["flags"]),
        "n_hallucinated": v["n_hallucinated"],
        "dose_groundedness": v["dose_groundedness"],
        "safety_reproduction": v["safety_reproduction"],
        "is_grounded": v["is_grounded"],
        "report_words": len(res["report"].split()),
        "gen_seconds": res["gen_seconds"],
    })

eval_df = pd.DataFrame(records)
eval_df.to_csv(OUT_DIR / "rag_end_to_end_evaluation.csv", index=False)

OUTCOME_LABELS = {
    "correct_treatment": "Correct treatment recommended",
    "wrong_treatment": "WRONG treatment recommended",
    "missed_treatment": "Abstained on a treatable case",
    "correct_no_treatment": "Correctly identified as needing no drug",
    "safe_abstention": "Abstained (no harm)",
}

print("\n" + "=" * 78)
print("  END-TO-END OUTCOMES")
print("=" * 78)
total = len(eval_df)
for key, label in OUTCOME_LABELS.items():
    n = int((eval_df["outcome"] == key).sum())
    if n:
        print(f"  {label:<44} {n:>4}  ({n / total * 100:5.1f}%)")
print("-" * 78)
wrong = int((eval_df["outcome"] == "wrong_treatment").sum())
covered = int((eval_df["status"] == "recommended").sum())
print(f"  Vision accuracy on these cases              "
      f"{eval_df['vision_correct'].mean() * 100:5.1f}%")
print(f"  Coverage (cases where a drug was proposed)  {covered / total * 100:5.1f}%")
print(f"  Wrong-treatment rate overall                {wrong / total * 100:5.1f}%")
print(f"  Wrong-treatment rate among covered cases    "
      f"{(wrong / covered * 100) if covered else 0:5.1f}%")
print("=" * 78)

print("\n  GROUNDEDNESS OF THE GENERATED REPORTS")
print("-" * 78)
print(f"  Generator                    {LLM.name}")
print(f"  Fully grounded reports       {eval_df['is_grounded'].mean() * 100:5.1f}%"
      f"   (all cases; abstentions are vacuously grounded)")
_rec = eval_df[eval_df["status"] == "recommended"]
if len(_rec):
    print(f"  Grounded among RECOMMENDED    {_rec['is_grounded'].mean() * 100:5.1f}%"
          f"   <- the number that matters, n={len(_rec)}")
    print(f"  Dose groundedness (recommended) {_rec['dose_groundedness'].mean() * 100:5.1f}%")
else:
    print("  No case cleared the confidence gate, so no drug-naming report was produced.")
print(f"  Reports naming a drug that was not retrieved  "
      f"{(eval_df['n_hallucinated'] > 0).mean() * 100:5.1f}%")
print(f"  Mean dose groundedness       {eval_df['dose_groundedness'].mean() * 100:5.1f}%")
print(f"  Mean safety reproduction     {eval_df['safety_reproduction'].mean() * 100:5.1f}%")
_ret = eval_df["retrieval_precision"].dropna()
if len(_ret):
    print(f"  Retrieval precision (retrieved drugs matching the TRUE disease) "
          f"{_ret.mean() * 100:5.1f}%")
print(f"  Mean report length           {eval_df['report_words'].mean():.0f} words")
print(f"  Mean generation time         {eval_df['gen_seconds'].mean():.2f} s")
print("-" * 78)

In [ ]:
# =============================================================================
#  CELL 20 - Error propagation and the risk-coverage trade-off
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Does a vision error become a treatment error?", fontsize=16, fontweight="bold")

# Panel 1: outcome breakdown
present = [(k, l) for k, l in OUTCOME_LABELS.items() if (eval_df["outcome"] == k).any()]
vals = [int((eval_df["outcome"] == k).sum()) for k, _ in present]
palette = {"correct_treatment": "#2E7D32", "correct_no_treatment": "#1565C0",
           "safe_abstention": "#42A5F5", "missed_treatment": "#EF6C00",
           "wrong_treatment": "#C62828"}
axes[0, 0].barh(range(len(vals)), vals, color=[palette[k] for k, _ in present])
axes[0, 0].set_yticks(range(len(vals)))
axes[0, 0].set_yticklabels([textwrap.fill(l, 28) for _, l in present], fontsize=9)
axes[0, 0].invert_yaxis()
axes[0, 0].set_xlabel("Cases")
axes[0, 0].set_title("End-to-end outcome distribution", fontweight="bold")
for i, v in enumerate(vals):
    axes[0, 0].text(v + 0.2, i, f"{v} ({v / total * 100:.0f}%)", va="center",
                    fontsize=9, fontweight="bold")

# Panel 2: risk-coverage curve
thresholds = np.linspace(0.0, 0.99, 60)
coverage, wrong_rate = [], []
for t in thresholds:
    covered_mask = (eval_df["confidence"] >= t) & \
                   (~eval_df["pred_disease"].isin(NO_TREATMENT))
    coverage.append(covered_mask.mean())
    if covered_mask.sum():
        wrong_rate.append((~eval_df.loc[covered_mask, "vision_correct"]).mean())
    else:
        wrong_rate.append(0.0)
axes[0, 1].plot(np.array(coverage) * 100, np.array(wrong_rate) * 100,
                "o-", lw=2.5, ms=4, color="#C62828")
current_cov = ((eval_df["confidence"] >= CFG.CONFIDENCE_THRESHOLD) &
               (~eval_df["pred_disease"].isin(NO_TREATMENT))).mean()
current_idx = int(np.argmin(np.abs(thresholds - CFG.CONFIDENCE_THRESHOLD)))
axes[0, 1].scatter([coverage[current_idx] * 100], [wrong_rate[current_idx] * 100],
                   s=240, marker="*", color="#1565C0", zorder=5,
                   label=f"operating point (T={CFG.CONFIDENCE_THRESHOLD})")
axes[0, 1].set_xlabel("Coverage: % of cases the system will act on")
axes[0, 1].set_ylabel("Wrong-treatment rate among those cases (%)")
axes[0, 1].set_title("Risk–coverage trade-off\n(move the threshold to buy safety with coverage)",
                     fontweight="bold")
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(alpha=0.3)

# Panel 3: confidence separates right from wrong
correct_conf = eval_df.loc[eval_df["vision_correct"], "confidence"]
wrong_conf = eval_df.loc[~eval_df["vision_correct"], "confidence"]
bins = np.linspace(0, 1, 16)
if len(correct_conf):
    axes[1, 0].hist(correct_conf, bins=bins, alpha=0.75, color="#2E7D32",
                    label="vision correct", edgecolor="white")
if len(wrong_conf):
    axes[1, 0].hist(wrong_conf, bins=bins, alpha=0.75, color="#C62828",
                    label="vision wrong", edgecolor="white")
axes[1, 0].axvline(CFG.CONFIDENCE_THRESHOLD, ls="--", color="black", lw=2,
                   label=f"abstention threshold")
axes[1, 0].set_xlabel("Calibrated confidence")
axes[1, 0].set_ylabel("Cases")
axes[1, 0].set_title("Is confidence usable as a safety gate?", fontweight="bold")
axes[1, 0].legend(fontsize=9)

# Panel 4: which confusions are clinically dangerous
wrong_cases = eval_df[eval_df["outcome"] == "wrong_treatment"]
if len(wrong_cases):
    pairs = Counter(zip(wrong_cases["true_disease"], wrong_cases["pred_disease"]))
    items = pairs.most_common(8)
    labels = [f"{t}\n-> treated as {p}" for (t, p), _ in items]
    axes[1, 1].barh(range(len(items)), [c for _, c in items], color="#C62828")
    axes[1, 1].set_yticks(range(len(items)))
    axes[1, 1].set_yticklabels(labels, fontsize=8)
    axes[1, 1].invert_yaxis()
    axes[1, 1].set_xlabel("Cases")
else:
    axes[1, 1].text(0.5, 0.5, "No wrong treatments\nin the evaluated sample",
                    ha="center", va="center", fontsize=13, color="#2E7D32",
                    fontweight="bold", transform=axes[1, 1].transAxes)
    axes[1, 1].set_xticks([])
    axes[1, 1].set_yticks([])
axes[1, 1].set_title("Mis-treatments by confusion pair", fontweight="bold")

plt.tight_layout()
savefig(fig, "rag_03_error_propagation")
plt.show()

risk_df = pd.DataFrame({"threshold": thresholds,
                        "coverage": coverage,
                        "wrong_treatment_rate": wrong_rate})
risk_df.to_csv(OUT_DIR / "rag_risk_coverage.csv", index=False)
print("\nRisk-coverage operating points:")
print(f"{'threshold':>10} {'coverage':>10} {'wrong rate':>12}")
for t in [0.0, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    i = int(np.argmin(np.abs(thresholds - t)))
    print(f"{thresholds[i]:>10.2f} {coverage[i] * 100:>9.1f}% {wrong_rate[i] * 100:>11.1f}%")

In [ ]:
# =============================================================================
#  CELL 21 - Does the LLM stay as grounded as the deterministic template?
# =============================================================================
if getattr(LLM, "is_llm", False):
    print("Comparing the LLM against the template baseline on identical inputs.\n")
    compare_rows = []
    n_compare = min(12, len(eval_indices))
    for n, idx in enumerate(tqdm(eval_indices[:n_compare], desc="LLM vs template")):
        img_t, lbl = test_ds[int(idx)]
        ctx = EVAL_CONTEXTS[n % len(EVAL_CONTEXTS)]
        for gen in (LLM, TEMPLATE_LLM):
            res = run_pipeline(img_t, context=ctx, generator=gen, true_label=lbl)
            v = res["verification"]
            compare_rows.append({
                "generator": "LLM" if gen is LLM else "Template",
                "grounded": v["is_grounded"],
                "hallucinated_drugs": v["n_hallucinated"],
                "dose_groundedness": v["dose_groundedness"],
                "safety_reproduction": v["safety_reproduction"],
                "words": len(res["report"].split()),
                "seconds": res["gen_seconds"],
            })
    compare_df = pd.DataFrame(compare_rows)
    summary = compare_df.groupby("generator").agg(
        grounded_pct=("grounded", lambda s: 100 * s.mean()),
        hallucinated_per_report=("hallucinated_drugs", "mean"),
        dose_groundedness=("dose_groundedness", "mean"),
        safety_reproduction=("safety_reproduction", "mean"),
        mean_words=("words", "mean"),
        mean_seconds=("seconds", "mean"),
    )
    print(summary.to_string(float_format=lambda v: f"{v:0.3f}"))
    summary.to_csv(OUT_DIR / "rag_llm_vs_template.csv")
    print("\nThe template cannot hallucinate by construction, so it is the ceiling for")
    print("groundedness and the floor for fluency. The gap is the price of natural language.")
else:
    compare_df = pd.DataFrame()
    print("Running with the template generator only - no LLM comparison to make.")
    print("Enable Internet and a GPU on Kaggle to load a real instruct model.")

In [ ]:
# =============================================================================
#  CELL 22 - Interactive: run the pipeline on any image with any context
# =============================================================================
def diagnose(image_path=None, image_index=None, **context):
    """Convenience entry point.

    diagnose(image_path='/kaggle/input/.../photo.jpg', pregnant=True)
    diagnose(image_index=3, infant=True, infected=True)
    """
    if image_path is not None:
        with Image.open(image_path) as im:
            img_t = EVAL_TF(im.convert("RGB"))
        true_label = None
    else:
        idx = 0 if image_index is None else int(image_index)
        img_t, true_label = test_ds[idx % len(test_ds)]
    return run_pipeline(img_t, context=context, true_label=true_label, verbose=True)


print("Try, for example:")
print("    diagnose(image_index=5, pregnant=True)")
print("    diagnose(image_index=12, infant=True, infected=True)")
print("    diagnose(image_path='/kaggle/input/<your-dataset>/<photo>.jpg', prefer_otc=True)\n")
_ = diagnose(image_index=int(eval_indices[0]), pregnant=True, pain=True)

In [ ]:
# =============================================================================
#  CELL 23 - Export
# =============================================================================
summary_payload = {
    "inputs": {
        "image_dataset": str(DATA_ROOT),
        "drug_spreadsheet": str(DRUG_FILE),
        "n_images": len(FILE_PATHS),
        "n_drug_cards": int(len(drugs_df)),
        "image_classes": CLASS_NAMES,
        "kb_diseases": KB_DISEASES,
        "classes_without_drugs": [c for c in CLASS_NAMES if c not in KB_DISEASES],
    },
    "vision": {
        "checkpoint": str(CKPT) if CKPT else "trained in this notebook",
        "test_accuracy": vision_acc,
        "test_macro_f1": float(vision_f1),
        "temperature": TEMPERATURE,
    },
    "retrieval": {"backend": retriever.backend, "top_k": CFG.TOP_K},
    "generator": LLM.name,
    "end_to_end": {
        "n_cases": int(len(eval_df)),
        "coverage": float((eval_df["status"] == "recommended").mean()),
        "wrong_treatment_rate": float((eval_df["outcome"] == "wrong_treatment").mean()),
        "grounded_report_rate": float(eval_df["is_grounded"].mean()),
        "mean_dose_groundedness": float(eval_df["dose_groundedness"].mean()),
        "mean_safety_reproduction": float(eval_df["safety_reproduction"].mean()),
    },
    "confidence_threshold": CFG.CONFIDENCE_THRESHOLD,
}

with open(OUT_DIR / "rag_summary.json", "w") as fh:
    json.dump(summary_payload, fh, indent=2, default=float)
drugs_df.drop(columns=["card"]).to_csv(OUT_DIR / "drug_knowledge_base.csv", index=False)

with open(OUT_DIR / "rag_example_reports.txt", "w") as fh:
    for i, res in enumerate(demo_results, 1):
        fh.write(f"{'=' * 84}\nCASE {i}: true={res['true_disease']} "
                 f"pred={res['disease']} ({res['confidence'] * 100:.1f}%)\n"
                 f"context={res['_context']}\n{'=' * 84}\n{res['report']}\n\n")

print("=" * 78)
print("  MULTIMODAL PIPELINE COMPLETE")
print("=" * 78)
print(f"  Images                     {len(FILE_PATHS)} across {NUM_CLASSES} classes")
print(f"  Drug cards                 {len(drugs_df)} across {len(KB_DISEASES)} diseases")
print(f"  Vision accuracy            {vision_acc * 100:.2f}%  (macro-F1 {vision_f1:.4f})")
print(f"  Retrieval backend          {retriever.backend}")
print(f"  Generator                  {LLM.name}")
print()
print(f"  Cases evaluated            {len(eval_df)}")
print(f"  Coverage                   "
      f"{(eval_df['status'] == 'recommended').mean() * 100:.1f}%")
print(f"  Wrong-treatment rate       "
      f"{(eval_df['outcome'] == 'wrong_treatment').mean() * 100:.1f}%")
print(f"  Fully grounded reports     {eval_df['is_grounded'].mean() * 100:.1f}%")
print(f"  Safety reproduction        {eval_df['safety_reproduction'].mean() * 100:.1f}%")
print()
print(f"  Figures  {len(list(FIG_DIR.glob('rag_*.png')))} -> {FIG_DIR}")
print(f"  Tables   {len(list(OUT_DIR.glob('rag_*.csv')))} CSV + rag_summary.json")
print("=" * 78)